# Duchenne Muscular Dystrophy: CRISPR-Corrected Isogenic Transcriptomics

## 01 — Data Loading, Quality Control and Differential Expression

Bulk RNA-seq analysis of a CRISPR-corrected isogenic DMD model (GEO [GSE189053](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE189053), Morera et al. 2022). A DMD patient-derived pluripotent stem cell line (**DMD-K2957fs**) is compared to its CRISPR-corrected isogenic control (**CORR-K2957fs**) across 5 time points of myogenic differentiation (Day -1, 0, 2, 5, 7), 4 replicates each.

This notebook loads and aggregates the raw Salmon quantifications, performs quality control, PCA, and per-day differential expression, exporting results for the functional enrichment analysis in `02_functional_enrichment.ipynb`.


In [1]:
import tarfile
from pathlib import Path

data_dir = Path("../data/raw")
raw_tar = data_dir / "GSE189053_RAW.tar"
extract_dir = data_dir / "salmon_quant"
extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(raw_tar, "r") as tar:
    tar.extractall(path=extract_dir)

sample_archives = sorted(extract_dir.glob("*.tar.gz"))
print("Sample archives found:", len(sample_archives))

for archive in sample_archives:
    sample_name = archive.name.split("_salmon.tar.gz")[0]
    sample_out_dir = extract_dir / sample_name
    sample_out_dir.mkdir(exist_ok=True)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path=sample_out_dir)

print("Extraction complete.")

C:\Users\javie\AppData\Local\Temp\ipykernel_15032\3067108805.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Sample archives found: 40


C:\Users\javie\AppData\Local\Temp\ipykernel_15032\3067108805.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=sample_out_dir)


Extraction complete.


In [2]:
import pandas as pd

quant_files = sorted(extract_dir.glob("*/*/quant.sf"))
print("quant.sf files found:", len(quant_files))

example = pd.read_csv(quant_files[0], sep="\t")
print(example.shape)
example.head()

quant.sf files found: 40
(227464, 5)


,Name,Length,EffectiveLength,TPM,NumReads
0,ENST00000456328.2,1657,1515.000,0.000000,0.000
1,ENST00000450305.2,632,490.000,0.000000,0.000
2,ENST00000488147.1,1351,1237.387,3.883596,63.507
3,ENST00000619216.1,68,8.000,0.000000,0.000
4,ENST00000473358.1,712,570.000,0.000000,0.000


## Transcript-to-Gene Aggregation

Salmon quantifies at the transcript level. Transcripts are mapped to gene symbols using `mygene` and aggregated by summing `NumReads` per gene, to build a gene x sample count matrix and load the corresponding sample metadata (genotype, replicate, day).


In [3]:
!pip install mygene

In [ ]:
import mygene

transcript_ids = example["Name"].str.split(".").str[0].unique().tolist()
print("Unique transcripts to map:", len(transcript_ids))

mg = mygene.MyGeneInfo()
results = mg.querymany(transcript_ids, scopes="ensembl.transcript", fields="symbol,ensembl.gene", species="human")

tx2gene = {}
for r in results:
    if "symbol" in r:
        tx2gene[r["query"]] = r["symbol"]

print("Transcripts successfully mapped:", len(tx2gene))

Unique transcripts to map: 227464


In [ ]:
tx2gene_series = pd.Series(tx2gene)

gene_counts = {}

for path in quant_files:
    sample_id = path.parts[-3].split("_")[0]
    df = pd.read_csv(path, sep="\t")
    df["transcript_id"] = df["Name"].str.split(".").str[0]
    df["gene"] = df["transcript_id"].map(tx2gene_series)
    df = df.dropna(subset=["gene"])
    gene_counts[sample_id] = df.groupby("gene")["NumReads"].sum()

counts_matrix = pd.DataFrame(gene_counts)
counts_matrix = counts_matrix.round().astype(int)

print(counts_matrix.shape)
counts_matrix.head()

In [ ]:
metadata_raw = [
    ("GSM5693941", "DMD", 1, -1), ("GSM5693942", "DMD", 1, 0), ("GSM5693943", "DMD", 1, 2), ("GSM5693944", "DMD", 1, 5), ("GSM5693945", "DMD", 1, 7),
    ("GSM5693946", "DMD", 2, -1), ("GSM5693947", "DMD", 2, 0), ("GSM5693948", "DMD", 2, 2), ("GSM5693949", "DMD", 2, 5), ("GSM5693950", "DMD", 2, 7),
    ("GSM5693951", "DMD", 3, -1), ("GSM5693952", "DMD", 3, 0), ("GSM5693953", "DMD", 3, 2), ("GSM5693954", "DMD", 3, 5), ("GSM5693955", "DMD", 3, 7),
    ("GSM5693956", "DMD", 4, -1), ("GSM5693957", "DMD", 4, 0), ("GSM5693958", "DMD", 4, 2), ("GSM5693959", "DMD", 4, 5), ("GSM5693960", "DMD", 4, 7),
    ("GSM5693961", "CORR", 1, -1), ("GSM5693962", "CORR", 1, 0), ("GSM5693963", "CORR", 1, 2), ("GSM5693964", "CORR", 1, 5), ("GSM5693965", "CORR", 1, 7),
    ("GSM5693966", "CORR", 2, -1), ("GSM5693967", "CORR", 2, 0), ("GSM5693968", "CORR", 2, 2), ("GSM5693969", "CORR", 2, 5), ("GSM5693970", "CORR", 2, 7),
    ("GSM5693971", "CORR", 3, -1), ("GSM5693972", "CORR", 3, 0), ("GSM5693973", "CORR", 3, 2), ("GSM5693974", "CORR", 3, 5), ("GSM5693975", "CORR", 3, 7),
    ("GSM5693976", "CORR", 4, -1), ("GSM5693977", "CORR", 4, 0), ("GSM5693978", "CORR", 4, 2), ("GSM5693979", "CORR", 4, 5), ("GSM5693980", "CORR", 4, 7),
]

metadata = pd.DataFrame(metadata_raw, columns=["sample_id", "genotype", "replicate", "day"])
metadata = metadata.set_index("sample_id")
metadata = metadata.loc[counts_matrix.columns]

metadata

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

counts_matrix.to_csv(processed_dir / "gene_counts_matrix.csv")
metadata.to_csv(processed_dir / "sample_metadata.csv")

## Quality Control and PCA

Library size and gene detection are checked across all 40 samples, followed by PCA on the 2,000 most variable genes to examine the global structure of the data with respect to genotype and differentiation day.


In [ ]:
library_sizes = counts_matrix.sum(axis=0)

qc_summary = metadata.copy()
qc_summary["library_size"] = library_sizes
qc_summary["n_genes_detected"] = (counts_matrix > 0).sum(axis=0)

qc_summary.sort_values("library_size")

In [ ]:
qc_summary.groupby("genotype")["library_size"].agg(["mean", "median", "std"])

In [ ]:
qc_summary.groupby("replicate")["library_size"].agg(["mean", "median", "std"])

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

log_counts = np.log2(counts_matrix + 1)

top_variable_genes = log_counts.var(axis=1).sort_values(ascending=False).head(2000).index
log_counts_hvg = log_counts.loc[top_variable_genes].T

scaler = StandardScaler()
scaled_data = scaler.fit_transform(log_counts_hvg)

pca = PCA(n_components=10)
pca_coords = pca.fit_transform(scaled_data)

pca_df = pd.DataFrame(
    pca_coords[:, :2],
    columns=["PC1", "PC2"],
    index=log_counts_hvg.index
)
pca_df = pca_df.join(metadata)

print("Variance explained by PC1 and PC2:", pca.explained_variance_ratio_[:2])
pca_df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))

colors = {"DMD": "#d62728", "CORR": "#1f77b4"}
markers = {-1: "o", 0: "s", 2: "^", 5: "D", 7: "*"}

for genotype in ["DMD", "CORR"]:
    for day in [-1, 0, 2, 5, 7]:
        subset = pca_df[(pca_df["genotype"] == genotype) & (pca_df["day"] == day)]
        ax.scatter(
            subset["PC1"],
            subset["PC2"],
            c=colors[genotype],
            marker=markers[day],
            s=80,
            label=f"{genotype} day {day}",
            edgecolor="black",
            linewidth=0.5
        )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA of gene expression: genotype and differentiation day")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("../results/figures/01_1_pca_genotype_day.png", dpi=150, bbox_inches="tight")
plt.show()

## Differential Expression by Time Point

DMD vs CORR differential expression is performed independently at each of the 5 time points using PyDESeq2, followed by volcano plots and a targeted check of the calcium-handling genes (ATP2A1, RYR1, CASQ2, SLN, CACNA1H) highlighted in the original study.


In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

de_results_by_day = {}

for day in [-1, 0, 2, 5, 7]:
    samples_day = metadata[metadata["day"] == day].index
    counts_day = counts_matrix[samples_day].T
    metadata_day = metadata.loc[samples_day]

    dds = DeseqDataSet(
        counts=counts_day,
        metadata=metadata_day,
        design_factors="genotype",
        refit_cooks=True
    )
    dds.deseq2()

    stat_res = DeseqStats(dds, contrast=["genotype", "DMD", "CORR"])
    stat_res.summary()

    de_results_by_day[day] = stat_res.results_df

    print(f"Day {day}: {(de_results_by_day[day]['padj'] < 0.05).sum()} genes with padj < 0.05")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), sharey=True)

days = [-1, 0, 2, 5, 7]

for ax, day in zip(axes, days):
    res = de_results_by_day[day].dropna(subset=["padj"])
    ax.scatter(
        res["log2FoldChange"],
        -np.log10(res["padj"].replace(0, 1e-300)),
        s=5,
        alpha=0.3,
        color="steelblue"
    )
    ax.axhline(-np.log10(0.05), color="red", linestyle="--", linewidth=1)
    ax.set_title(f"Day {day}")
    ax.set_xlabel("log2 fold change (DMD vs CORR)")

axes[0].set_ylabel("-log10(padj)")

plt.tight_layout()
plt.savefig("../results/figures/01_2_volcano_by_day.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
outlier_day2 = de_results_by_day[2].dropna(subset=["padj"]).sort_values("padj").head(5)
outlier_day2

In [ ]:
calcium_genes = ["ATP2A1", "RYR1", "CASQ2", "SLN", "CACNA1H"]

calcium_summary = pd.DataFrame({
    day: {gene: de_results_by_day[day].loc[gene, "log2FoldChange"] if gene in de_results_by_day[day].index else None for gene in calcium_genes}
    for day in days
}).T

calcium_summary.index.name = "day"
calcium_summary

In [ ]:
calcium_padj = pd.DataFrame({
    day: {gene: de_results_by_day[day].loc[gene, "padj"] if gene in de_results_by_day[day].index else None for gene in calcium_genes}
    for day in days
}).T

calcium_padj.index.name = "day"
calcium_padj

Results are exported for use in the functional enrichment notebook.


In [ ]:
for day, results in de_results_by_day.items():
    results.to_csv(processed_dir / f"de_results_day{day}.csv")